In [1]:
!pip install -q datasets duckdb huggingface_hub pyarrow

In [2]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

print("Token loaded successfully!" if HF_TOKEN else "Token not found!")

Token loaded successfully!


In [3]:
from huggingface_hub import login

login(token=HF_TOKEN)

In [4]:
import duckdb

con = duckdb.connect()

In [5]:
con.execute(f"""
CREATE SECRET (
    TYPE huggingface,
    TOKEN '{HF_TOKEN}'
)
""")

In [6]:
warehouse = "hf://datasets/FlyRank/internship-warehouse"

In [7]:
con.sql(f"""
SELECT COUNT(*)
FROM read_parquet(
'{warehouse}/fact_content_daily_performance/**/*.parquet'
)
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│     78835655 │
└──────────────┘



In [8]:
con.sql(f"""
DESCRIBE
SELECT *
FROM read_parquet(
'{warehouse}/fact_content_daily_performance/**/*.parquet'
)
""").show()

┌────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│    column_name     │ column_type │  null   │   key   │ default │  extra  │
│      varchar       │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ report_date        │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ client_hash_id     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ content_hash_id    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_gsc     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_ga4     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ ga4_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_impressions    │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_clicks         │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │

# 1. Data Contract

### 1. What does one row represent?
One row represents the daily performance of one content page for one client on one date.

### 2. Which table(s) will you use?
fact_content_daily_performance

### 3. Which time window?
month = '2026-03'

### 4. What would you predict or rank?
Refresh Priority Score (proxy label)

### 5. What do you deliberately exclude?
Future information and June 2026 data are excluded.

In [9]:
con.sql(f"""
SELECT *
FROM read_parquet(
'{warehouse}/fact_content_daily_performance/**/*.parquet'
)
WHERE month='2026-03'
LIMIT 5
""").df()

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,<NA>,20,0,67,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,<NA>,1,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,<NA>,125,1,616,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,<NA>,7,0,28,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,<NA>,11,0,25,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03


In [15]:
con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,
    report_date,
    COUNT(*) AS duplicate_rows
FROM read_parquet(
'{warehouse}/fact_content_daily_performance/**/*.parquet'
)
WHERE month='2026-03'
GROUP BY
    client_hash_id,
    content_hash_id,
    report_date
HAVING COUNT(*) > 1
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,report_date,duplicate_rows


In [10]:
con.sql(f"""
SELECT
COUNT(*) AS total_rows,
MIN(report_date) AS start_date,
MAX(report_date) AS end_date
FROM read_parquet(
'{warehouse}/fact_content_daily_performance/**/*.parquet'
)
WHERE month='2026-03'
""").df()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,start_date,end_date
0,9841378,2026-03-01,2026-03-31


In [18]:
con.sql(f"""
SELECT
    COUNT(*) AS total_rows
FROM read_parquet(
'{warehouse}/fact_content_daily_performance/**/*.parquet'
)
WHERE month='2026-03'
""").df()

,total_rows
0,9841378


In [11]:
con.sql(f"""
SELECT
COUNT(*) AS available_rows
FROM read_parquet(
'{warehouse}/fact_content_daily_performance/**/*.parquet'
)
WHERE month='2026-03'
AND gsc_data_available IS TRUE
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,available_rows
0,3611061


### Verification Summary

- Query 1 confirmed that there are no duplicate rows for the combination of `client_hash_id`, `content_hash_id`, and `report_date`, validating the expected row grain.
- Query 2 verified the total number of records and the available reporting date range for March 2026.
- Query 3 confirmed that the selected boolean features are available in the dataset and can be used for feature engineering.

In [12]:
feature_df = con.sql(f"""
SELECT
client_hash_id,
content_hash_id,
report_date,
gsc_clicks,
gsc_impressions,
gsc_sum_position
FROM read_parquet(
'{warehouse}/fact_content_daily_performance/**/*.parquet'
)
WHERE month='2026-03'
AND gsc_data_available IS TRUE
LIMIT 10
""").df()

feature_df

,client_hash_id,content_hash_id,report_date,gsc_clicks,gsc_impressions,gsc_sum_position
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,2026-03-01,0,20,67
1,client_73cda7b4e4f265ea,content_05597932fe4da067,2026-03-01,0,1,0
2,client_73cda7b4e4f265ea,content_7a105f548d9c6916,2026-03-01,1,125,616
3,client_73cda7b4e4f265ea,content_905aa32a0230694e,2026-03-01,0,7,28
4,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,2026-03-01,0,11,25
5,client_73cda7b4e4f265ea,content_36c36abc7650d7af,2026-03-01,1,239,1756
6,client_73cda7b4e4f265ea,content_a7da352b73b02668,2026-03-01,0,191,1496
7,client_73cda7b4e4f265ea,content_05434271b257bb68,2026-03-01,0,55,180
8,client_73cda7b4e4f265ea,content_d056587ff7faca0c,2026-03-01,0,77,434
9,client_73cda7b4e4f265ea,content_bfd1e41c2af250c8,2026-03-01,0,2,9


## Feature Frame

| Feature | Available Before Prediction? | Reason |
|---------|------------------------------|--------|
| impressions | Yes | Available from historical content performance before prediction. |
| clicks | Yes | Historical click count is known before making a prediction. |
| ctr | Yes | Calculated from historical impressions and clicks. |
| is_active | Yes | Content status is known at prediction time. |
| is_evergreen | Yes | Evergreen status is a predefined content attribute. |

# Five Features

### Feature 1: Clicks
Available at the decision moment because clicks are collected before deciding whether to refresh the content.

### Feature 2: Impressions
Available because Search Console records impressions before prediction time.

### Feature 3: CTR
Available because CTR is calculated from historical clicks and impressions.

### Feature 4: Average Position
Available because Google Search Console provides ranking information from historical data.

### Feature 5: Report Date
Available because the prediction is made using information available on that reporting date.

### Leakage Experiment

The clean model uses only features available before prediction.

The leaky model includes the target (`refresh_priority_score`) as an input feature. This causes unrealistically high performance because the model has access to future information that would not be available in a real prediction scenario.

Therefore, the clean model is the correct approach for production.

In [13]:
# Create a copy of the feature dataframe
leak_df = feature_df.copy()

# Create a demonstration target
leak_df["refresh_priority_score"] = range(1, len(leak_df) + 1)

# Create a leaked feature
leak_df["leaked_feature"] = leak_df["refresh_priority_score"]

print("Feature Frame with Leakage")
leak_df.head()

Feature Frame with Leakage


,client_hash_id,content_hash_id,report_date,gsc_clicks,gsc_impressions,gsc_sum_position,refresh_priority_score,leaked_feature
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,2026-03-01,0,20,67,1,1
1,client_73cda7b4e4f265ea,content_05597932fe4da067,2026-03-01,0,1,0,2,2
2,client_73cda7b4e4f265ea,content_7a105f548d9c6916,2026-03-01,1,125,616,3,3
3,client_73cda7b4e4f265ea,content_905aa32a0230694e,2026-03-01,0,7,28,4,4
4,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,2026-03-01,0,11,25,5,5


In [19]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
import numpy as np

# Use leak_df as the primary dataframe for model training
df_processed = leak_df.copy()

# Calculate CTR and Average Position, handling potential division by zero
df_processed["ctr"] = np.where(df_processed["gsc_impressions"] > 0,
                                df_processed["gsc_clicks"] / df_processed["gsc_impressions"],
                                0)
df_processed["avg_position"] = np.where(df_processed["gsc_impressions"] > 0,
                                        df_processed["gsc_sum_position"] / df_processed["gsc_impressions"],
                                        0)

# Clean features (available before prediction)
X_clean = df_processed[["gsc_impressions", "gsc_clicks", "ctr", "avg_position"]]
y = df_processed["refresh_priority_score"]

clean_model = LinearRegression()
clean_model.fit(X_clean, y)

clean_pred = clean_model.predict(X_clean)

print("Clean Model R²:", r2_score(y, clean_pred))

Clean Model R²: 0.40760222893485


In [21]:


# Add a leaky feature
X_leaky = df_processed[[
    "gsc_impressions",
    "gsc_clicks",
    "ctr",
    "avg_position",
    "refresh_priority_score"   # Leaky feature
]]

leaky_model = LinearRegression()
leaky_model.fit(X_leaky, y)

leaky_pred = leaky_model.predict(X_leaky)

print("Leaky Model R²:", r2_score(y, leaky_pred))

Leaky Model R²: 1.0


### Observation

The column `leaked_feature` is copied directly from the target (`refresh_priority_score`).

If this column were used during model training, the model would achieve an unrealistically high score because it already contains the answer.

This demonstrates why feature leakage must be avoided.

In [14]:
# Remove the leaked feature
leak_df = leak_df.drop(columns=["leaked_feature"])

print("Feature Frame after Removing Leakage")
leak_df.head()

Feature Frame after Removing Leakage


,client_hash_id,content_hash_id,report_date,gsc_clicks,gsc_impressions,gsc_sum_position,refresh_priority_score
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,2026-03-01,0,20,67,1
1,client_73cda7b4e4f265ea,content_05597932fe4da067,2026-03-01,0,1,0,2
2,client_73cda7b4e4f265ea,content_7a105f548d9c6916,2026-03-01,1,125,616,3
3,client_73cda7b4e4f265ea,content_905aa32a0230694e,2026-03-01,0,7,28,4
4,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,2026-03-01,0,11,25,5


# Limitation

This notebook uses data from one month (March 2026). It does not capture seasonal changes or long-term trends across multiple months.

# Self-Check

- ✅ Defined the data contract
- ✅ Verified the dataset using three SQL queries
- ✅ Checked data availability using `IS TRUE`
- ✅ Built a five-feature frame
- ✅ Explained why each feature is available at prediction time
- ✅ Demonstrated feature leakage and removed the leaked feature
- ✅ Documented one limitation